### Tools

Models can request to call tools that performs tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
    1. A schema, including the name of the tool, a description, and/or argument definitions(aften a JSON schema)
    2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["MISTRAL_API_KEY"]=os.getenv("MISTRAL_API_KEY")
model = init_chat_model("mistral-small-latest")
response = model.invoke("Why we need Generative AI in Agentic AI?")
response

AIMessage(content='Generative AI plays a **critical role in Agentic AI** by enabling autonomous agents to interact with the real world, make decisions, and perform complex tasks. Here’s why Generative AI is essential for Agentic AI:\n\n### **1. Enables Natural Language Understanding & Generation**\nAgentic AI agents need to **understand human instructions, generate responses, and communicate effectively**—something Generative AI (LLMs like LLama, Mistral, etc.) excels at.\n- **Example:** A customer service agent uses Generative AI to interpret user queries and respond in natural language.\n- **Why?** Unlike traditional rule-based systems, Generative AI can handle **ambiguity, context, and nuance** in human language.\n\n### **2. Powers Autonomous Decision-Making**\nAgentic AI agents must **perceive their environment, reason, and take actions**—Generative AI helps by:\n- **Generating hypotheses** (e.g., predicting possible outcomes of different actions).\n- **Simulating scenarios** (e.g.

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'D2qA9R75t', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, 'index': 0}]} response_metadata={'token_usage': {'prompt_tokens': 83, 'total_tokens': 95, 'completion_tokens': 12, 'prompt_tokens_details': {'cached_tokens': 0}, 'service_tier': 'standard'}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'} id='lc_run--019fffe6-6e9d-79f2-b8e8-23db714e88c4-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'D2qA9R75t', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 83, 'output_tokens': 12, 'total_tokens': 95}
Tool: get_weather
Args: {'location': 'Boston'}


### Tools Execution Loops 

In [4]:
messages = [{"role":"user", "content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response)

content='The weather in **Boston** is currently **sunny**. Enjoy the day! ☀️' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 101, 'total_tokens': 123, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}, 'service_tier': 'standard'}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'} id='lc_run--019fffe6-7fb4-7231-909f-3960f40b8b31-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 101, 'output_tokens': 22, 'total_tokens': 123}


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pmcwn14VF', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 82, 'total_tokens': 94, 'completion_tokens': 12, 'prompt_tokens_details': {'cached_tokens': 0}, 'service_tier': 'standard'}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019fffe6-7d25-72e1-b72b-3f46177bfe43-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'pmcwn14VF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 82, 'output_tokens': 12, 'total_tokens': 94}),
 ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='pmcwn14VF')]